<a href="https://colab.research.google.com/github/ChaMooKwan/Secure-SDL-JWT-Storage-methods-research/blob/main/Automative_Searching_for_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from dotenv import load_dotenv

# .env 파일을 읽어 환경 변수로 설정
load_dotenv()

# os.get env를 통해 사용
GITHUB_TOKEN = os.getenv("KEY")

In [ ]:
import requests
import time
import sys

# [중요] GitHub Personal Access Token (PAT)을 발급받아 여기에 입력하세요.
# 토큰 없이 Search API를 사용하면 몇 번의 요청만으로 Rate Limit에 걸립니다.
#GITHUB_TOKEN = "YOUR_GITHUB_TOKEN_HERE"
HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

def search_repositories(query, max_results=100):
    """지정된 쿼리로 GitHub 저장소를 검색합니다."""
    repos = []
    page = 1

    while len(repos) < max_results:
        url = f"https://api.github.com/search/repositories?q={query}&per_page=100&page={page}"
        response = requests.get(url, headers=HEADERS)

        if response.status_code == 200:
            items = response.json().get('items', [])
            if not items:
                break # 더 이상 결과가 없으면 종료
            repos.extend(items)
            page += 1
            time.sleep(2) # Secondary Rate Limit 방지
        else:
            print(f"[!] 저장소 검색 에러: {response.status_code} - {response.text}")
            break

    return repos[:max_results]

def search_code_in_repo(repo_full_name, keyword):
    """특정 저장소 내에서 코드를 검색합니다."""
    # GitHub Code Search API는 저장소를 지정(repo:name)해야 효과적입니다.
    query = f"{keyword} repo:{repo_full_name}"
    url = f"https://api.github.com/search/code?q={query}"

    response = requests.get(url, headers=HEADERS)

    if response.status_code == 200:
        items = response.json().get('items', [])
        return len(items) > 0
    elif response.status_code == 403:
        # Search API는 분당 30회 제한 등 매우 엄격합니다. 403 발생 시 백오프 대기가 필요합니다.
        print(f"\n[!] Rate Limit 초과. 60초 대기 후 재시도합니다...")
        time.sleep(60)
        return search_code_in_repo(repo_full_name, keyword)
    else:
        return False

def main():
    if GITHUB_TOKEN == "YOUR_GITHUB_TOKEN_HERE":
        print("실행 전 GITHUB_TOKEN을 입력해주세요.")
        sys.exit(1)

    # 1. 1차 필터링: 쇼핑몰/이커머스 관련이면서 JWT를 언급하는 저장소 검색
    # 언어를 지정하면(예: language:javascript) 결과의 정확도를 높일 수 있습니다.
    repo_query = "e-commerce OR shopping-mall jwt in:readme,description language:j;t"
    print(f"[*] 1차 필터링 검색어: {repo_query}")

    # 여유 있게 200개 정도의 후보군을 먼저 가져옵니다.
    candidate_repos = search_repositories(repo_query, max_results=200)
    print(f"[*] 총 {len(candidate_repos)}개의 후보 저장소를 찾았습니다.\n")

    valid_targets = []
    target_count = 100

    # 2. 2차 필터링: 각 후보 저장소의 코드 레벨 스캔
    # 하드코딩된 시크릿 키나 취약하게 구현되었을 확률이 높은 패턴을 검색합니다.
    # 예: 'jwt.sign' 함수와 'secret'이라는 단어가 같은 파일에 존재하는지 스캔
    code_keyword = '"jwt.sign" secret'

    print("[*] 2차 코드 스캔을 시작합니다. (GitHub API 제한으로 시간이 소요될 수 있습니다)")

    for repo in candidate_repos:
        repo_name = repo['full_name']
        sys.stdout.write(f"[-] 스캔 중: {repo_name} ... ")
        sys.stdout.flush()

        # GitHub Search API 남용을 막기 위한 필수 딜레이 (매우 중요)
        time.sleep(3)

        is_jwt_used = search_code_in_repo(repo_name, code_keyword)

        if is_jwt_used:
            print(" [발견!]")
            valid_targets.append(repo_name)
        else:
            print(" [없음]")

        if len(valid_targets) >= target_count:
            print("\n[*] 목표한 100개의 시스템을 찾았습니다!")
            break

    print("\n=== 최종 스캔 완료 결과 ===")
    for i, target in enumerate(valid_targets, 1):
        print(f"{i}. https://github.com/{target}")

if __name__ == "__main__":
    main()

[*] 1차 필터링 검색어: e-commerce OR shopping-mall jwt in:readme,description language:javascript
[*] 총 200개의 후보 저장소를 찾았습니다.

[*] 2차 코드 스캔을 시작합니다. (GitHub API 제한으로 시간이 소요될 수 있습니다)
[-] 스캔 중: john-smilga/node-express-course ...  [발견!]
[-] 스캔 중: meabhisingh/mernProjectEcommerce ...  [발견!]
[-] 스캔 중: burakorkmez/mern-ecommerce ...  [발견!]
[-] 스캔 중: aimeos/aimeos ...  [없음]
[-] 스캔 중: coderdost/MERN-ecommerce-Frontend ...  [없음]
[-] 스캔 중: SajidAnTechie/ShopPoint ...  [발견!]
[-] 스캔 중: yoonic/atlas ...  [없음]
[-] 스캔 중: imranhsayed/gatsby-woocommerce-themes ...  [없음]
[-] 스캔 중: coderdost/MERN-ecommerce-backend ...  [발견!]
[-] 스캔 중: Saurabh-8585/MERN-E-Commerce-Frontend ... 
[!] Rate Limit 초과. 60초 대기 후 재시도합니다...
 [없음]
[-] 스캔 중: BMayhew/awesome-sites-to-test-on ...  [없음]
[-] 스캔 중: huanghanzhilian/c-shopping ...  [발견!]
[-] 스캔 중: shubham1710/MERN-E-Commerce ...  [발견!]
[-] 스캔 중: postmanlabs/e-commerce-store-express ...  [발견!]
[-] 스캔 중: basir/node-react-ecommerce ...  [발견!]
[-] 스캔 중: riteshk-007/nextjs-store ...  [발